# 🎯 Ćwiczenie 3: MapReduce — Analiza recenzji filmowych

**Kurs:** Python w Zastosowaniach Big Data
**Zajęcia nr:** 3
**Szacowany czas:** ⏱️ ~90 minut
**Poziom trudności:** ⭐⭐⭐ (Średni)

## 📚 Cele nauki

Po ukończeniu tego notebooka będziesz potrafić:
1. Zaimplementować ręcznie fazy **map**, **shuffle** i **reduce** w czystym Pythonie
2. Policzyć częstość słów osobno dla recenzji pozytywnych i negatywnych
3. Napisać generyczną funkcję w stylu `reduceByKey`
4. Połączyć statystykę opisową (z Wykładu 1) z analizą tekstu
5. Zrównoleglić prosty word count przy pomocy `concurrent.futures`

## 📋 Wymagania wstępne
- Podstawy Pythona: listy, słowniki, funkcje, list comprehensions
- Podstawy statystyki opisowej (średnia, mediana, odchylenie standardowe — z Wykładu 1)
- Znajomość pojęć **MapReduce** z Wykładu 3 (map / shuffle / reduce)

---
> 💡 **Tip:** Uruchamiaj każdą komórkę przez `Shift+Enter` lub przycisk ▶️.
> Jeśli utkniesz, sprawdź komórki z podpowiedziami (💡 Hint) albo zapytaj prowadzącego!

In [ ]:
# @title 🔧 Setup — generowanie syntetycznych recenzji filmowych

import random
import re
from collections import defaultdict

import pandas as pd

random.seed(42)  # 👈 Reprodukowalność — wszyscy dostaną te same dane!

# --- Słowniki budulcowe do generowania recenzji ---
podmioty = [
    "film", "aktorstwo", "fabuła", "scenariusz", "muzyka",
    "reżyseria", "montaż", "scenografia", "gra aktorska", "zakończenie",
]

przymiotniki_pozytywne = [
    "świetny", "fenomenalny", "wciągający", "genialny", "znakomity",
    "poruszający", "zaskakujący", "znakomicie zrealizowany", "niesamowity", "wzruszający",
]

przymiotniki_negatywne = [
    "nudny", "słaby", "przewidywalny", "rozczarowujący", "chaotyczny",
    "płytki", "toporny", "denny", "żenujący", "bezsensowny",
]

frazy_pozytywne = [
    "Polecam każdemu, kto lubi dobre kino.",
    "Nie mogłem oderwać wzroku od ekranu.",
    "To jeden z najlepszych filmów tego roku.",
    "Wyszedłem z kina zachwycony.",
    "Świetna rozrywka na wieczór z przyjaciółmi.",
]

frazy_negatywne = [
    "Szkoda czasu i pieniędzy na bilet.",
    "Zasnąłem w połowie seansu.",
    "To jeden z najgorszych filmów, jakie widziałem.",
    "Wyszedłem z kina rozczarowany.",
    "Nie polecam nawet w deszczowy dzień.",
]


def wygeneruj_recenzje(n: int) -> list[dict]:
    """Generuje n syntetycznych recenzji filmowych z etykietą sentymentu."""
    recenzje = []
    for i in range(n):
        sentyment = random.choice(["pozytywna", "negatywna"])
        if sentyment == "pozytywna":
            przymiotniki, frazy = przymiotniki_pozytywne, frazy_pozytywne
        else:
            przymiotniki, frazy = przymiotniki_negatywne, frazy_negatywne

        liczba_zdan = random.randint(2, 4)
        zdania = []
        for _ in range(liczba_zdan):
            podmiot = random.choice(podmioty)
            przymiotnik = random.choice(przymiotniki)
            zdania.append(f"{podmiot.capitalize()} był {przymiotnik}.")
        zdania.append(random.choice(frazy))
        tekst = " ".join(zdania)

        recenzje.append({
            "review_id": i,
            "text": tekst,
            "sentiment": sentyment,
            "word_count": len(tekst.split()),
        })
    return recenzje


recenzje = wygeneruj_recenzje(2000)
df = pd.DataFrame(recenzje)

print(f"✅ Wygenerowano {len(df)} recenzji")
print(f"📊 Rozkład sentymentu:\n{df['sentiment'].value_counts()}")
df.head()

## 📖 Szybkie przypomnienie — MapReduce

Na Wykładzie 3 poznaliśmy paradygmat **MapReduce**:

| Faza | Co robi | W naszym przykładzie |
|------|---------|----------------------|
| **Map** | Przetwarza każdy rekord niezależnie, emituje pary `(klucz, wartość)` | Dla każdego słowa w recenzji emitujemy `(słowo, 1)` |
| **Shuffle** | Grupuje wartości po kluczu | Grupujemy wszystkie `1` dla tego samego słowa razem |
| **Reduce** | Agreguje wartości per klucz | Sumujemy `1`-ki, żeby dostać częstość słowa |

Dziś zaimplementujemy to **od zera** w czystym Pythonie, na naszych 2000 syntetycznych recenzjach filmowych — a potem wykorzystamy tę intuicję do analizy sentymentu.

## 🎓 Guided Example — manualny MapReduce word count

**Zadanie:** Policzmy, ile razy każde słowo występuje we wszystkich recenzjach razem — implementując ręcznie fazy map, shuffle i reduce.

In [ ]:
# @title 🎓 Guided Example — Uruchom i przeanalizuj

def map_step(teksty: list[str]) -> list[tuple[str, int]]:
    """Faza MAP: dla każdego słowa w każdym tekście emitujemy (słowo, 1)."""
    wyniki = []
    for tekst in teksty:
        slowa = re.findall(r"[a-ząćęłńóśźż]+", tekst.lower())  # 👈 tylko litery, bez interpunkcji
        wyniki.extend((slowo, 1) for slowo in slowa)
    return wyniki


def shuffle_step(pary: list[tuple[str, int]]) -> dict[str, list[int]]:
    """Faza SHUFFLE: grupujemy wartości po kluczu (słowie)."""
    grupy: dict[str, list[int]] = defaultdict(list)
    for klucz, wartosc in pary:
        grupy[klucz].append(wartosc)
    return grupy


def reduce_step(grupy: dict[str, list[int]]) -> dict[str, int]:
    """Faza REDUCE: sumujemy wartości dla każdego klucza."""
    return {klucz: sum(wartosci) for klucz, wartosci in grupy.items()}


def map_reduce_word_count(teksty: list[str]) -> dict[str, int]:
    """Pełny pipeline MapReduce: map -> shuffle -> reduce."""
    mapped = map_step(teksty)
    grouped = shuffle_step(mapped)
    return reduce_step(grouped)


wszystkie_teksty = df["text"].tolist()
czestosc_slow = map_reduce_word_count(wszystkie_teksty)

print(f"📊 Liczba unikalnych słów: {len(czestosc_slow)}")
print(f"🔝 5 najczęstszych słów: {sorted(czestosc_slow.items(), key=lambda x: -x[1])[:5]}")

---

## ✏️ Ćwiczenie 1: Częstość słów per klasa sentymentu

**Trudność:** ⭐

Wykorzystując funkcję `map_reduce_word_count()` z guided example, policz częstość słów **osobno** dla recenzji pozytywnych i osobno dla negatywnych.

**Oczekiwany wynik:** dwa słowniki: `czestosc_pozytywne` i `czestosc_negatywne`.

> 💡 **Hint:** Użyj `df[df["sentiment"] == "pozytywna"]["text"].tolist()`, żeby wyciągnąć tylko teksty pozytywnych recenzji.

In [ ]:
# ✏️ Ćwiczenie 1: Częstość słów per klasa sentymentu

# TODO: Wyciągnij teksty recenzji pozytywnych do listy
teksty_pozytywne = ...  # Zastąp ... swoim kodem

# TODO: Wyciągnij teksty recenzji negatywnych do listy
teksty_negatywne = ...  # Zastąp ... swoim kodem

# TODO: Policz częstość słów dla każdej klasy używając map_reduce_word_count()
czestosc_pozytywne = ...  # Zastąp ... swoim kodem
czestosc_negatywne = ...  # Zastąp ... swoim kodem

print(f"Unikalne słowa (pozytywne): {len(czestosc_pozytywne)}")
print(f"Unikalne słowa (negatywne): {len(czestosc_negatywne)}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 1 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: filtrujemy DataFrame po sentymencie i wyciągamy kolumnę text jako listę napisów
teksty_pozytywne = df[df["sentiment"] == "pozytywna"]["text"].tolist()
teksty_negatywne = df[df["sentiment"] == "negatywna"]["text"].tolist()

# Krok 2: korzystamy z gotowego pipeline'u map -> shuffle -> reduce z guided example
czestosc_pozytywne = map_reduce_word_count(teksty_pozytywne)
czestosc_negatywne = map_reduce_word_count(teksty_negatywne)

print(f"Unikalne słowa (pozytywne): {len(czestosc_pozytywne)}")
print(f"Unikalne słowa (negatywne): {len(czestosc_negatywne)}")

In [ ]:
# @title ✅ Test — Ćwiczenie 1 (uruchom, żeby sprawdzić odpowiedź)

try:
    assert isinstance(teksty_pozytywne, list), "❌ teksty_pozytywne powinno być listą"
    assert isinstance(teksty_negatywne, list), "❌ teksty_negatywne powinno być listą"
    assert len(teksty_pozytywne) + len(teksty_negatywne) == len(df), "❌ Suma powinna dać wszystkie recenzje"
    assert isinstance(czestosc_pozytywne, dict), "❌ czestosc_pozytywne powinno być słownikiem"
    assert isinstance(czestosc_negatywne, dict), "❌ czestosc_negatywne powinno być słownikiem"
    assert "świetny" in czestosc_pozytywne, "❌ Brakuje oczywistego pozytywnego słowa — sprawdź filtrowanie"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")
except Exception as e:
    print(f"❌ Coś poszło nie tak: {e}")

---

## ✏️ Ćwiczenie 2: Top-N najczęstszych słów kluczowych

**Trudność:** ⭐⭐

Napisz funkcję `top_n_slow(czestosc: dict[str, int], n: int) -> list[tuple[str, int]]`, która zwraca `n` najczęściej występujących słów, posortowanych malejąco po liczbie wystąpień.

**Oczekiwany wynik:** lista krotek `(słowo, liczba_wystapien)` o długości `n`.

> 💡 **Hint:** `sorted(iterable, key=..., reverse=True)` to Twój przyjaciel.

In [ ]:
# ✏️ Ćwiczenie 2: Top-N najczęstszych słów kluczowych

def top_n_slow(czestosc: dict[str, int], n: int) -> list[tuple[str, int]]:
    """Zwraca n najczęściej występujących słów, malejąco."""
    # TODO: posortuj czestosc.items() po wartości malejąco i zwróć pierwsze n
    ...


top_10_pozytywne = top_n_slow(czestosc_pozytywne, 10)
top_10_negatywne = top_n_slow(czestosc_negatywne, 10)

print(f"Top 10 (pozytywne): {top_10_pozytywne}")
print(f"Top 10 (negatywne): {top_10_negatywne}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 2 (kliknij, aby zobaczyć krok po kroku)

def top_n_slow(czestosc: dict[str, int], n: int) -> list[tuple[str, int]]:
    """Zwraca n najczęściej występujących słów, malejąco."""
    # Krok 1: sortujemy pary (słowo, liczba_wystapien) po liczbie wystąpień malejąco
    # Krok 2: bierzemy pierwsze n elementów posortowanej listy
    return sorted(czestosc.items(), key=lambda para: -para[1])[:n]


top_10_pozytywne = top_n_slow(czestosc_pozytywne, 10)
top_10_negatywne = top_n_slow(czestosc_negatywne, 10)

print(f"Top 10 (pozytywne): {top_10_pozytywne}")
print(f"Top 10 (negatywne): {top_10_negatywne}")

In [ ]:
# @title ✅ Test — Ćwiczenie 2 (uruchom, żeby sprawdzić odpowiedź)

try:
    assert len(top_10_pozytywne) == 10, f"❌ Oczekiwano 10 elementów, otrzymano {len(top_10_pozytywne)}"
    liczby = [liczba for _, liczba in top_10_pozytywne]
    assert liczby == sorted(liczby, reverse=True), "❌ Wynik powinien być posortowany malejąco"
    assert top_n_slow(czestosc_pozytywne, 3) == top_10_pozytywne[:3], "❌ top_n_slow powinno być spójne dla różnych n"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")
except Exception as e:
    print(f"❌ Coś poszło nie tak: {e}")

---

## ✏️ Ćwiczenie 3: Generyczna funkcja `reduce_by_key()`

**Trudność:** ⭐⭐

W prawdziwych frameworkach (Spark) istnieje operacja `reduceByKey(funkcja)`, która grupuje pary `(klucz, wartość)` i agreguje wartości dowolną funkcją — nie tylko sumą!

Napisz generyczną funkcję `reduce_by_key(pary, funkcja_agregujaca)`, która przyjmuje listę par `(klucz, wartość)` oraz dowolną funkcję agregującą listę wartości (np. `sum`, `max`, `len`, albo własną funkcję liczącą średnią).

**Oczekiwany wynik:** słownik `{klucz: wynik_agregacji}`.

> 💡 **Hint:** Wykorzystaj `shuffle_step()` z guided example, żeby najpierw pogrupować wartości po kluczu.

In [ ]:
# ✏️ Ćwiczenie 3: Generyczna funkcja reduce_by_key()

def reduce_by_key(pary: list[tuple[str, int]], funkcja_agregujaca) -> dict[str, int]:
    """Generyczny odpowiednik Sparkowego reduceByKey — grupuje po kluczu
    i agreguje wartości dowolną funkcją (sum, max, len, ...)."""
    # TODO: pogrupuj pary po kluczu (możesz użyć shuffle_step)
    grupy = ...

    # TODO: dla każdego klucza zastosuj funkcja_agregujaca na liście jego wartości
    wynik = ...
    return wynik


pary_slow_pozytywne = map_step(teksty_pozytywne)

# Test 1: suma (powinno dać ten sam wynik co map_reduce_word_count)
suma_wystapien = reduce_by_key(pary_slow_pozytywne, sum)

# Test 2: liczba wystąpień to samo co suma dla par (słowo, 1) — ale spróbujmy max
maks_wystapien = reduce_by_key(pary_slow_pozytywne, max)

print(f"Suma wystąpień słowa 'świetny': {suma_wystapien.get('świetny')}")
print(f"Maks wystąpień (pojedynczy wiersz) słowa 'świetny': {maks_wystapien.get('świetny')}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 3 (kliknij, aby zobaczyć krok po kroku)

def reduce_by_key(pary: list[tuple[str, int]], funkcja_agregujaca) -> dict[str, int]:
    """Generyczny odpowiednik Sparkowego reduceByKey — grupuje po kluczu
    i agreguje wartości dowolną funkcją (sum, max, len, ...)."""
    # Krok 1 (SHUFFLE): grupujemy wartości po kluczu, wykorzystując gotową funkcję z guided example
    grupy = shuffle_step(pary)

    # Krok 2 (REDUCE): na liście wartości każdego klucza stosujemy przekazaną funkcję agregującą
    wynik = {klucz: funkcja_agregujaca(wartosci) for klucz, wartosci in grupy.items()}
    return wynik


pary_slow_pozytywne = map_step(teksty_pozytywne)

# Test 1: suma (powinno dać ten sam wynik co map_reduce_word_count)
suma_wystapien = reduce_by_key(pary_slow_pozytywne, sum)

# Test 2: liczba wystąpień to samo co suma dla par (słowo, 1) — ale spróbujmy max
maks_wystapien = reduce_by_key(pary_slow_pozytywne, max)

print(f"Suma wystąpień słowa 'świetny': {suma_wystapien.get('świetny')}")
print(f"Maks wystąpień (pojedynczy wiersz) słowa 'świetny': {maks_wystapien.get('świetny')}")

In [ ]:
# @title ✅ Test — Ćwiczenie 3 (uruchom, żeby sprawdzić odpowiedź)

try:
    assert isinstance(suma_wystapien, dict), "❌ Wynik reduce_by_key powinien być słownikiem"
    assert suma_wystapien == czestosc_pozytywne, "❌ reduce_by_key(pary, sum) powinno dać ten sam wynik co map_reduce_word_count"
    assert all(v == 1 for v in maks_wystapien.values()), "❌ max z listy samych 1-ek powinien zawsze dać 1"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")
except Exception as e:
    print(f"❌ Coś poszło nie tak: {e}")

---

## ✏️ Ćwiczenie 4: Rozkład długości recenzji (statystyka opisowa)

**Trudność:** ⭐⭐

Wracamy do Wykładu 1! Kolumna `word_count` w `df` mówi, ile słów ma każda recenzja. Policz podstawowe statystyki opisowe długości recenzji — **osobno** dla pozytywnych i negatywnych.

**Oczekiwany wynik:** dwa słowniki `statystyki_pozytywne` i `statystyki_negatywne`, każdy z kluczami `"srednia"`, `"mediana"`, `"odchylenie_std"`.

> 💡 **Hint:** `df["word_count"].mean()`, `.median()`, `.std()` — to dokładnie to, co robiliśmy na Wykładzie 1!

In [ ]:
# ✏️ Ćwiczenie 4: Rozkład długości recenzji

def policz_statystyki(dlugosci: pd.Series) -> dict[str, float]:
    """Zwraca słownik ze średnią, medianą i odchyleniem standardowym."""
    # TODO: oblicz średnią, medianę i odchylenie standardowe kolumny dlugosci
    return {
        "srednia": ...,
        "mediana": ...,
        "odchylenie_std": ...,
    }


# TODO: wywołaj policz_statystyki() dla word_count recenzji pozytywnych i negatywnych
statystyki_pozytywne = ...
statystyki_negatywne = ...

print(f"Pozytywne: {statystyki_pozytywne}")
print(f"Negatywne: {statystyki_negatywne}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 4 (kliknij, aby zobaczyć krok po kroku)

def policz_statystyki(dlugosci: pd.Series) -> dict[str, float]:
    """Zwraca słownik ze średnią, medianą i odchyleniem standardowym."""
    # Krok 1-3: standardowe miary tendencji centralnej i rozproszenia z Wykładu 1
    return {
        "srednia": dlugosci.mean(),
        "mediana": dlugosci.median(),
        "odchylenie_std": dlugosci.std(),
    }


# Krok 4: liczymy statystyki osobno dla word_count recenzji pozytywnych i negatywnych
statystyki_pozytywne = policz_statystyki(df[df["sentiment"] == "pozytywna"]["word_count"])
statystyki_negatywne = policz_statystyki(df[df["sentiment"] == "negatywna"]["word_count"])

print(f"Pozytywne: {statystyki_pozytywne}")
print(f"Negatywne: {statystyki_negatywne}")

In [ ]:
# @title ✅ Test — Ćwiczenie 4 (uruchom, żeby sprawdzić odpowiedź)

try:
    for statystyki in (statystyki_pozytywne, statystyki_negatywne):
        assert set(statystyki.keys()) == {"srednia", "mediana", "odchylenie_std"}, "❌ Brakuje wymaganych kluczy"
        assert statystyki["srednia"] > 0, "❌ Średnia długość recenzji powinna być dodatnia"
    oczekiwana_srednia = df[df["sentiment"] == "pozytywna"]["word_count"].mean()
    assert abs(statystyki_pozytywne["srednia"] - oczekiwana_srednia) < 0.01, "❌ Niepoprawna średnia"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")
except Exception as e:
    print(f"❌ Coś poszło nie tak: {e}")

---

## ✏️ Ćwiczenie 5: Słownictwo unikalne dla każdego sentymentu

**Trudność:** ⭐⭐⭐

Wykorzystując `set()`, znajdź słowa, które pojawiają się **tylko** w recenzjach pozytywnych (a nigdy w negatywnych) oraz odwrotnie.

**Oczekiwany wynik:** dwa zbiory: `unikalne_pozytywne` i `unikalne_negatywne`.

> 💡 **Hint:** Operator różnicy zbiorów w Pythonie to `-`, np. `zbior_a - zbior_b`.

In [ ]:
# ✏️ Ćwiczenie 5: Słownictwo unikalne dla każdego sentymentu

# TODO: zbuduj zbiory słów kluczowych (kluczy słowników czestosc_*)
zbior_pozytywne = ...
zbior_negatywne = ...

# TODO: użyj różnicy zbiorów, żeby znaleźć słowa unikalne dla każdej klasy
unikalne_pozytywne = ...
unikalne_negatywne = ...

print(f"Słowa tylko w pozytywnych: {sorted(unikalne_pozytywne)[:10]}")
print(f"Słowa tylko w negatywnych: {sorted(unikalne_negatywne)[:10]}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 5 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: klucze słowników częstości to właśnie zbiory unikalnych słów użytych w danej klasie
zbior_pozytywne = set(czestosc_pozytywne.keys())
zbior_negatywne = set(czestosc_negatywne.keys())

# Krok 2: operator różnicy zbiorów `-` daje elementy występujące tylko po jednej stronie
unikalne_pozytywne = zbior_pozytywne - zbior_negatywne
unikalne_negatywne = zbior_negatywne - zbior_pozytywne

print(f"Słowa tylko w pozytywnych: {sorted(unikalne_pozytywne)[:10]}")
print(f"Słowa tylko w negatywnych: {sorted(unikalne_negatywne)[:10]}")

In [ ]:
# @title ✅ Test — Ćwiczenie 5 (uruchom, żeby sprawdzić odpowiedź)

try:
    assert isinstance(unikalne_pozytywne, set), "❌ unikalne_pozytywne powinno być zbiorem (set)"
    assert isinstance(unikalne_negatywne, set), "❌ unikalne_negatywne powinno być zbiorem (set)"
    assert unikalne_pozytywne.isdisjoint(unikalne_negatywne), "❌ Te zbiory nie powinny mieć części wspólnej"
    assert "świetny" in unikalne_pozytywne, "❌ 'świetny' powinno być unikalne dla recenzji pozytywnych"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")
except Exception as e:
    print(f"❌ Coś poszło nie tak: {e}")

---

## ✏️ Ćwiczenie 6: Filtrowanie stopwords

**Trudność:** ⭐⭐

Słowa takie jak "był", "i", "w" są bardzo częste, ale nie niosą informacji o sentymencie (to tzw. **stopwords**). Napisz funkcję `filtruj_stopwords()`, która usuwa je ze słownika częstości, i sprawdź, jak zmienia się Top-10.

**Oczekiwany wynik:** funkcja `filtruj_stopwords(czestosc: dict, stopwords: set) -> dict` oraz nowy Top-10 po filtracji.

> 💡 **Hint:** Dict comprehension z warunkiem: `{k: v for k, v in d.items() if k not in stopwords}`.

In [ ]:
# ✏️ Ćwiczenie 6: Filtrowanie stopwords

stopwords_pl = {"był", "była", "było", "i", "w", "z", "na", "nawet"}


def filtruj_stopwords(czestosc: dict[str, int], stopwords: set[str]) -> dict[str, int]:
    """Usuwa stopwords ze słownika częstości słów."""
    # TODO: zwróć nowy słownik bez kluczy znajdujących się w stopwords
    ...


czestosc_pozytywne_czyste = filtruj_stopwords(czestosc_pozytywne, stopwords_pl)
top_10_czyste = top_n_slow(czestosc_pozytywne_czyste, 10)

print(f"Top 10 po filtracji stopwords: {top_10_czyste}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 6 (kliknij, aby zobaczyć krok po kroku)

stopwords_pl = {"był", "była", "było", "i", "w", "z", "na", "nawet"}


def filtruj_stopwords(czestosc: dict[str, int], stopwords: set[str]) -> dict[str, int]:
    """Usuwa stopwords ze słownika częstości słów."""
    # Krok 1: dict comprehension z warunkiem — zachowujemy tylko klucze spoza stopwords
    return {slowo: liczba for slowo, liczba in czestosc.items() if slowo not in stopwords}


czestosc_pozytywne_czyste = filtruj_stopwords(czestosc_pozytywne, stopwords_pl)
top_10_czyste = top_n_slow(czestosc_pozytywne_czyste, 10)

print(f"Top 10 po filtracji stopwords: {top_10_czyste}")

In [ ]:
# @title ✅ Test — Ćwiczenie 6 (uruchom, żeby sprawdzić odpowiedź)

try:
    assert "był" not in czestosc_pozytywne_czyste, "❌ 'był' powinno zostać odfiltrowane jako stopword"
    assert len(czestosc_pozytywne_czyste) < len(czestosc_pozytywne), "❌ Po filtracji powinno zostać mniej słów"
    assert all(slowo not in stopwords_pl for slowo, _ in top_10_czyste), "❌ Top 10 nie powinno zawierać stopwords"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")
except Exception as e:
    print(f"❌ Coś poszło nie tak: {e}")

---

## ✏️ Ćwiczenie 7: Ranking recenzji po liczbie słów kluczowych

**Trudność:** ⭐⭐⭐

Mając zbiór słów kluczowych (np. `unikalne_pozytywne` z Ćwiczenia 5), napisz funkcję, która dla każdej recenzji liczy, ile z tych słów kluczowych zawiera, a następnie zwraca **5 recenzji** z największą liczbą trafień.

**Oczekiwany wynik:** lista 5 krotek `(review_id, liczba_trafien)`, posortowana malejąco.

> 💡 **Hint:** Dla każdej recenzji zamień tekst na zbiór słów (`set(tekst.lower().split())`) i policz `len(zbior_slow & slowa_kluczowe)`.

In [ ]:
# ✏️ Ćwiczenie 7: Ranking recenzji po liczbie słów kluczowych

def zlicz_trafienia(tekst: str, slowa_kluczowe: set[str]) -> int:
    """Liczy, ile słów kluczowych występuje w danym tekście."""
    # TODO: zamień tekst na zbiór słów i policz część wspólną ze slowa_kluczowe
    ...


# TODO: dla recenzji pozytywnych policz trafienia względem unikalne_pozytywne
# Zbuduj listę krotek (review_id, liczba_trafien) i posortuj malejąco, weź top 5
recenzje_pozytywne = df[df["sentiment"] == "pozytywna"]
trafienia = ...
top_5_recenzji = ...

print(f"Top 5 recenzji wg trafień: {top_5_recenzji}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 7 (kliknij, aby zobaczyć krok po kroku)

def zlicz_trafienia(tekst: str, slowa_kluczowe: set[str]) -> int:
    """Liczy, ile słów kluczowych występuje w danym tekście."""
    # Krok 1: zamieniamy tekst na zbiór słów (tanie porównanie przez przecięcie zbiorów)
    # Krok 2: len() przecięcia zbiorów to liczba trafień słów kluczowych
    return len(set(tekst.lower().split()) & slowa_kluczowe)


recenzje_pozytywne = df[df["sentiment"] == "pozytywna"]
# Krok 3: dla każdej recenzji liczymy trafienia względem zbioru słów unikalnych dla pozytywnych
trafienia = [
    (review_id, zlicz_trafienia(tekst, unikalne_pozytywne))
    for review_id, tekst in zip(recenzje_pozytywne["review_id"], recenzje_pozytywne["text"])
]
# Krok 4: sortujemy malejąco po liczbie trafień i bierzemy 5 najlepszych
top_5_recenzji = sorted(trafienia, key=lambda para: -para[1])[:5]

print(f"Top 5 recenzji wg trafień: {top_5_recenzji}")

In [ ]:
# @title ✅ Test — Ćwiczenie 7 (uruchom, żeby sprawdzić odpowiedź)

try:
    assert len(top_5_recenzji) == 5, f"❌ Oczekiwano 5 recenzji, otrzymano {len(top_5_recenzji)}"
    liczby_trafien = [n for _, n in top_5_recenzji]
    assert liczby_trafien == sorted(liczby_trafien, reverse=True), "❌ Wynik powinien być posortowany malejąco"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")
except Exception as e:
    print(f"❌ Coś poszło nie tak: {e}")

---

## 🏆 Bonus Challenge (opcjonalnie): Zrównoleglenie word count

**Trudność:** ⭐⭐⭐⭐

Prawdziwa moc MapReduce bierze się z **równoległości** — faza `map` działa jednocześnie na wielu maszynach. Zasymuluj to lokalnie, korzystając z `concurrent.futures.ProcessPoolExecutor`:

1. Podziel listę recenzji na **4 fragmenty** (symulując 4 "węzły" klastra)
2. Uruchom `map_step()` na każdym fragmencie **równolegle**, w osobnym procesie
3. Połącz wyniki (`shuffle_step` + `reduce_step`) i porównaj z wynikiem sekwencyjnym — powinny być identyczne!
4. Zmierz czas wykonania wersji sekwencyjnej vs równoległej (`time.perf_counter()`)

> ⚠️ Na małych danych wersja równoległa może być **wolniejsza** przez narzut na tworzenie procesów — to normalne i dokładnie taki sam trade-off, jaki opisywaliśmy dla MapReduce na wykładzie!

In [ ]:
# 🏆 Bonus: Zrównoleglenie word count przy pomocy concurrent.futures

import time
from concurrent.futures import ProcessPoolExecutor

# TODO: podziel wszystkie_teksty na 4 równe fragmenty (listy list[str])
fragmenty = ...

start_sekwencyjny = time.perf_counter()
wynik_sekwencyjny = map_reduce_word_count(wszystkie_teksty)
czas_sekwencyjny = time.perf_counter() - start_sekwencyjny

# TODO: uruchom map_step na każdym fragmencie równolegle przez ProcessPoolExecutor,
# połącz wszystkie wyniki (listy par) w jedną listę, a następnie shuffle_step + reduce_step
start_rownolegly = time.perf_counter()
with ProcessPoolExecutor(max_workers=4) as executor:
    wyniki_map = ...  # np. list(executor.map(map_step, fragmenty))

wszystkie_pary = ...  # spłaszcz listę list par
wynik_rownolegly = reduce_step(shuffle_step(wszystkie_pary))
czas_rownolegly = time.perf_counter() - start_rownolegly

print(f"Sekwencyjnie: {czas_sekwencyjny:.4f}s")
print(f"Równolegle:   {czas_rownolegly:.4f}s")
print(f"Wyniki identyczne: {wynik_sekwencyjny == wynik_rownolegly}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Bonus: zrównoleglenie word count (kliknij, aby zobaczyć krok po kroku)

# Krok 1: dzielimy listę tekstów na 4 w miarę równe fragmenty, symulując 4 węzły klastra
# (bez numpy: zaokrąglenie w górę przy pomocy całkowitego dzielenia)
rozmiar_fragmentu = (len(wszystkie_teksty) + 3) // 4
fragmenty = [wszystkie_teksty[i:i + rozmiar_fragmentu] for i in range(0, len(wszystkie_teksty), rozmiar_fragmentu)]

start_sekwencyjny = time.perf_counter()
wynik_sekwencyjny = map_reduce_word_count(wszystkie_teksty)
czas_sekwencyjny = time.perf_counter() - start_sekwencyjny

# Krok 2 (MAP równoległy): każdy fragment jest przetwarzany niezależnie w osobnym procesie
start_rownolegly = time.perf_counter()
with ProcessPoolExecutor(max_workers=4) as executor:
    wyniki_map = list(executor.map(map_step, fragmenty))

# Krok 3: spłaszczamy listę list par do jednej listy, a potem robimy shuffle + reduce (jak w guided example)
wszystkie_pary = [para for fragment_wynikow in wyniki_map for para in fragment_wynikow]
wynik_rownolegly = reduce_step(shuffle_step(wszystkie_pary))
czas_rownolegly = time.perf_counter() - start_rownolegly

print(f"Sekwencyjnie: {czas_sekwencyjny:.4f}s")
print(f"Równolegle:   {czas_rownolegly:.4f}s")
print(f"Wyniki identyczne: {wynik_sekwencyjny == wynik_rownolegly}")

---

## 📝 Podsumowanie

Świetna robota! 🎉 Oto, co dziś przećwiczyłeś/przećwiczyłaś:

- ✅ Ręczna implementacja faz **map**, **shuffle** i **reduce** w czystym Pythonie
- ✅ Analiza częstości słów osobno dla różnych klas (sentyment pozytywny/negatywny)
- ✅ Generyczna funkcja `reduce_by_key()` — dokładnie ten sam koncept co Sparkowe `reduceByKey`
- ✅ Połączenie statystyki opisowej (Wykład 1) z analizą tekstu
- ✅ Filtrowanie stopwords i budowanie rankingów
- 🏆 (Bonus) Zrównoleglenie word count przez `concurrent.futures`

### 📚 Dalsze materiały
- [Dean & Ghemawat — MapReduce: Simplified Data Processing on Large Clusters (2004)](https://research.google/pubs/pub62/)
- [Dokumentacja `concurrent.futures`](https://docs.python.org/3/library/concurrent.futures.html)

### 🔮 Co dalej?
Na **Wykładzie 4** poznamy **Apache Spark** i zobaczymy, jak dokładnie te same koncepcje (map, reduce, partycjonowanie danych) wyglądają w PySparku — tylko dużo szybciej i wygodniej! 🚀